# 19A4B — Frozen Cycle-25 AIA Inference: 2021

## Purpose

Run **frozen inference only** for the 2021 portion of the untouched Cycle-25 test set.

This notebook does **not**:
- retrain the CNN-GRU;
- refit calibration;
- reselect the threshold;
- compute annual performance metrics;
- use 2021 results for any tuning decision.

It only:
1. verifies the staged 2021 payload;
2. loads the frozen 19A2 base model;
3. applies the frozen 19A3 Platt calibrator;
4. applies the frozen 19A3 operating threshold;
5. saves one prediction row per frozen 2021 target.

Final Cycle-25 metrics are deferred until all 2021–2025 predictions have been concatenated and verified.


In [ ]:
from pathlib import Path
import hashlib, json, sys, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

HOME = Path.home()

TARGET_MAP = HOME / "aia19_cycle25_staging_prep" / "target_maps" / "cycle25_2021_targets_local_paths.csv.gz"

BASE = HOME / "aia19_cnn_gru_20260917"
MODEL_PATH = BASE / "models" / "cnn_gru_cycle24_final_refit.pt"
NORM_PATH = BASE / "normalisation.json"

CAL_DIR = HOME / "aia19_calibration_threshold_20260918"
CAL_PATH = CAL_DIR / "platt_calibrator.json"
THR_PATH = CAL_DIR / "operating_threshold.json"

OUT = HOME / "aia19_cycle25_predictions"
OUT.mkdir(parents=True, exist_ok=True)

YEAR = 2021
EXPECTED_TARGETS = 5104
EXPECTED_POSITIVES = 126
EXPECTED_UNIQUE_OBJECTS = 5516

EXPECTED_MODEL_SHA256 = "11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76"
EXPECTED_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia335"]

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
DROPOUT = 0.30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for p in [TARGET_MAP, MODEL_PATH, NORM_PATH, CAL_PATH, THR_PATH]:
    assert p.exists(), p

print("DEVICE:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 1. Verify frozen model identity and 2021 target map

In [ ]:
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

model_sha = sha256_file(MODEL_PATH)
print("MODEL_SHA256:", model_sha)
assert model_sha == EXPECTED_MODEL_SHA256

df = pd.read_csv(TARGET_MAP)

assert len(df) == EXPECTED_TARGETS
assert int(df["label_48h_final"].sum()) == EXPECTED_POSITIVES
assert set(df["stored_year"].unique()) == {YEAR}
assert set(df["role"].unique()) == {"independent_cycle25_test"}

print("Targets:", len(df))
print("Positives:", int(df["label_48h_final"].sum()))
print("Regions:", df["region_component_id"].nunique())


## 2. Verify the staged payload before inference

In [ ]:
LOCAL_COLS = ["local_tminus288", "local_tminus192", "local_tminus96"]

all_paths = pd.unique(pd.concat([df[c] for c in LOCAL_COLS], ignore_index=True))
assert len(all_paths) == EXPECTED_UNIQUE_OBJECTS, len(all_paths)

missing = []
zero = []

for p in all_paths:
    q = Path(p)
    if not q.exists():
        missing.append(str(q))
    elif q.stat().st_size <= 0:
        zero.append(str(q))

print("Expected unique staged objects:", EXPECTED_UNIQUE_OBJECTS)
print("Existing:", EXPECTED_UNIQUE_OBJECTS - len(missing))
print("Missing:", len(missing))
print("Zero-byte:", len(zero))

if missing or zero:
    raise RuntimeError("2021 staged payload integrity failed.")

# Canary-read one object before full inference.
with np.load(all_paths[0], allow_pickle=False) as z:
    x0 = z["x"]
    ch0 = [str(v) for v in z["channels"].tolist()]
    assert x0.shape == (512,512,6)
    assert x0.dtype == np.float32
    assert ch0 == EXPECTED_CHANNELS
    assert np.isfinite(x0).all()

print("2021 payload integrity/readability PASS")


## 3. Load frozen preprocessing and frozen 19A3 post-processing

In [ ]:
norm = json.loads(NORM_PATH.read_text())
assert norm["channel_order"] == EXPECTED_CHANNELS

channel_scale = np.asarray(norm["channel_scale"], dtype=np.float32)
channel_mean = np.asarray(norm["channel_mean"], dtype=np.float32)
channel_std = np.asarray(norm["channel_std"], dtype=np.float32)

cal = json.loads(CAL_PATH.read_text())
thr = json.loads(THR_PATH.read_text())

PLATT_COEF = float(cal["coefficient"])
PLATT_INTERCEPT = float(cal["intercept"])
FROZEN_THRESHOLD = float(thr["threshold"])

print("Platt coefficient:", PLATT_COEF)
print("Platt intercept:", PLATT_INTERCEPT)
print("Frozen threshold:", FROZEN_THRESHOLD)

assert abs(PLATT_COEF - 0.5846760189574352) < 1e-12
assert abs(PLATT_INTERCEPT - (-3.6728404851652265)) < 1e-12
assert abs(FROZEN_THRESHOLD - 0.030438695842933242) < 1e-15


## 4. Rebuild the exact frozen CNN-GRU

In [ ]:
class TemporalAIADataset(Dataset):
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        with np.load(path, allow_pickle=False) as z:
            x = z["x"]
            channels = [str(v) for v in z["channels"].tolist()]

            if x.shape != (512,512,6):
                raise ValueError((path, x.shape))
            if channels != EXPECTED_CHANNELS:
                raise ValueError((path, channels))
            if not np.isfinite(x).all():
                raise ValueError(f"Nonfinite tensor: {path}")

            x = x.astype(np.float32, copy=False)

        x = np.arcsinh(x / channel_scale.reshape(1,1,6))
        x = (x - channel_mean.reshape(1,1,6)) / channel_std.reshape(1,1,6)

        t = torch.from_numpy(x).permute(2,0,1).contiguous()
        t = F.interpolate(
            t.unsqueeze(0),
            size=(IMAGE_SIZE, IMAGE_SIZE),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)
        return t

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = torch.stack([
            self._load(r["local_tminus288"]),
            self._load(r["local_tminus192"]),
            self._load(r["local_tminus96"]),
        ])
        return (
            x,
            int(r["label_48h_final"]),
            r["target_sample_id"],
            r["region_component_id"],
            int(r["HARPNUM"]),
        )

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, drop=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin,cout,3,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout,cout,3,padding=1,bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Dropout2d(drop) if drop else nn.Identity(),
        )
    def forward(self, x):
        return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed=256):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(6,32,.05),
            ConvBlock(32,64,.05),
            ConvBlock(64,128,.10),
            ConvBlock(128,192,.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192,embed),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
    def forward(self, x):
        return self.proj(self.enc(x))

class Model(nn.Module):
    def __init__(self, embed=256, hidden=192):
        super().__init__()
        self.frame = FrameCNN(embed)
        self.gru = nn.GRU(embed, hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden,1),
        )
    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)

model = Model().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("Frozen model loaded.")


## 5. Run frozen inference — no metrics

In [ ]:
loader = DataLoader(
    TemporalAIADataset(df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=False,
)

records = []
t0 = time.time()

with torch.no_grad():
    for bi, (x, y, sid, region, harpnum) in enumerate(loader, 1):
        logits = model(x.to(DEVICE, non_blocking=True)).cpu().numpy().astype(float)

        raw_p = 1.0 / (1.0 + np.exp(-logits))
        cal_logit = PLATT_COEF * logits + PLATT_INTERCEPT
        cal_p = 1.0 / (1.0 + np.exp(-cal_logit))
        pred = (cal_p >= FROZEN_THRESHOLD).astype(int)

        for j in range(len(logits)):
            records.append({
                "target_sample_id": sid[j],
                "stored_year": YEAR,
                "region_component_id": region[j],
                "HARPNUM": int(harpnum[j]),
                "y_true": int(y[j]),
                "raw_logit": float(logits[j]),
                "raw_probability": float(raw_p[j]),
                "calibrated_probability": float(cal_p[j]),
                "frozen_threshold": FROZEN_THRESHOLD,
                "frozen_prediction": int(pred[j]),
            })

        if bi % 100 == 0 or bi == len(loader):
            print(f"2021 inference batch {bi}/{len(loader)}", flush=True)

pred_df = pd.DataFrame(records)

assert len(pred_df) == EXPECTED_TARGETS
assert pred_df["target_sample_id"].is_unique
assert int(pred_df["y_true"].sum()) == EXPECTED_POSITIVES
assert np.isfinite(pred_df[["raw_logit","raw_probability","calibrated_probability"]].to_numpy()).all()

elapsed = time.time() - t0
print("Inference rows:", len(pred_df))
print("Elapsed seconds:", elapsed)
print("No performance metrics computed.")


## 6. Preserve the 2021 prediction shard and protocol evidence

In [ ]:
PRED_PATH = OUT / "cycle25_2021_frozen_predictions.csv.gz"
pred_df.to_csv(PRED_PATH, index=False, compression="gzip")

pred_sha = sha256_file(PRED_PATH)

protocol = {
    "status": "CYCLE25_2021_FROZEN_AIA_PREDICTION_SHARD_COMPLETE_NO_METRICS",
    "year": YEAR,
    "targets": int(len(pred_df)),
    "positives": int(pred_df["y_true"].sum()),
    "base_model_sha256": model_sha,
    "normalisation_path": str(NORM_PATH),
    "platt_coefficient": PLATT_COEF,
    "platt_intercept": PLATT_INTERCEPT,
    "frozen_threshold": FROZEN_THRESHOLD,
    "prediction_file": str(PRED_PATH),
    "prediction_sha256": pred_sha,
    "model_weights_updated": False,
    "calibrator_refit": False,
    "threshold_reselected": False,
    "performance_metrics_computed": False,
    "cycle25_used_for_tuning": False,
    "scientific_clearance": False,
}

PROTO_PATH = OUT / "cycle25_2021_protocol_record.json"
PROTO_PATH.write_text(json.dumps(protocol, indent=2) + "\n")

print(json.dumps(protocol, indent=2))
print("PREDICTION_SHA256:", pred_sha)
print("2021_FROZEN_INFERENCE_COMPLETE")


## Handoff

Before deleting the 2021 local AIA payload:

1. confirm `cycle25_2021_frozen_predictions.csv.gz` exists;
2. confirm it contains exactly 5,104 unique target rows;
3. preserve its SHA256 and protocol record;
4. preferably copy the prediction shard/protocol to the Git checkpoint area;
5. only then delete `/mnt/disks/aia-cache/cycle25_yearwise/2021`.

No 2021 performance score is to be inspected or used for decisions.
